# The Bootstrap

### Estimating the Accuracy of a Statistic of Interest

One of the great advantages of the bootstrap approach is that it can
be applied in almost all situations. No complicated mathematical
calculations are required. While there are several implementations
of the bootstrap in Python, its use for estimating
standard error is simple enough that we write our own function
below for the case when our data is stored
in a dataframe.

To illustrate the bootstrap, we
start with a simple example.
The  `Portfolio`  data set in the `ISLP` package is described
in Section 5.2. The goal is to estimate the
sampling variance of the parameter $\alpha$ given in formula 5.6:
\begin{equation*}
 \alpha=\frac{\sigma_Y^2-\sigma_{X Y}}{\sigma_X^2+\sigma_Y^2-2 \sigma_{X Y}}.
\end{equation*}
We will
create a function
`alpha_func()`, which takes as input a dataframe `D` assumed
to have columns `X` and `Y`, as well as a
vector `idx` indicating which observations should be used to
estimate 
$\alpha$. The function then outputs the estimate for $\alpha$ based on
the selected observations.

In [24]:
import numpy as np
from scipy.stats import norm
from ISLP import load_data

In [25]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
   cov_ = np.cov(D[['X','Y']].loc[idx], rowvar=False)
   return ((cov_[1,1] - cov_[0,1]) /
           (cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

This function returns an estimate for $\alpha$
based on applying the minimum
    variance formula 5.6 to the observations indexed by
the argument `idx`.  For instance, the following command
estimates $\alpha$ using all 100 observations.

In [26]:
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

Next we randomly select
100 observations from `range(100)`, with replacement. This is equivalent
to constructing a new bootstrap data set and recomputing $\hat{\alpha}$
based on the new data set.

In [27]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
           rng.choice(100,
                      100,
                      replace=True))

np.float64(0.6074452469619004)

This process can be generalized to create a simple function `boot_SE()` for
computing the bootstrap standard error for arbitrary
functions that take only a data frame as an argument.

In [57]:
def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    n = n or D.shape[0]
    values=[]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)

        # ENTER YOUR CODE HERE
        values.append(value)

    
    values = np.array(values)
    return np.std(values, ddof=1)
    



Notice the use of `_` as a loop variable in `for _ in range(B)`. This is often used if the value of the counter is
unimportant and simply makes sure  the loop is executed `B` times.

Let’s use our function to evaluate the accuracy of our
estimate of $\alpha$ using $B=1{,}000$ bootstrap replications. 

In [58]:
alpha_SE = boot_SE(alpha_func,
                   Portfolio,
                   B=1000,
                   seed=0)
alpha_SE

np.float64(0.09122739031705925)

The final output should show that the bootstrap estimate for ${\rm SE}(\hat{\alpha})$ is $0.0912$.

## Bootstrap Confidence Interval

In the previous sections, we explored the use of the nonparametric bootstrap to estimate the sampling variability of a statistic by resampling directly from the observed data. This nonparametric approach requires no assumptions about the underlying data distribution, making it a flexible and widely applicable method for assessing the variability of estimates. Building on this foundation, we will now examine different ways to utilize the bootstrap, particularly focusing on methods for constructing confidence intervals. We will also explore several alternative bootstrap methods that can be applied to different data scenarios, each offering unique advantages and addressing specific data characteristics.

### Standard Normal Bootstrap Confidence Interval

The **Standard Normal Bootstrap Confidence Interval** is a straightforward approach for constructing confidence intervals, based on the assumption of normality. Suppose $\hat{\theta}$ is an estimator of the parameter $\theta$, with estimated standard error $se(\hat{\theta})$. If $\hat{\theta}$ is a sample mean and the sample size is large, the Central Limit Theorem implies that

\begin{equation*}
Z = \frac{\hat{\theta} - E[\hat{\theta}]}{se(\hat{\theta})}
\end{equation*}

is approximately standard normal. If $\hat{\theta}$ is unbiased for $\theta$, then an approximate $100(1 - \alpha)\%$ confidence interval for $\theta$ is given by

\begin{equation*}
\hat{\theta} \pm z_{\alpha / 2} \, se(\hat{\theta}),
\end{equation*}

where $z_{\alpha / 2} = \Phi^{-1}(1 - \alpha / 2)$. While this method is simple, it relies on assumptions of normality and unbiasedness, and it treats $se(\hat{\theta})$ as a known parameter. Here, we estimate $se(\hat{\theta})$ using the bootstrap replicates of $\hat{\theta}$.


In [59]:
Auto = load_data('Auto')
mpg_data = Auto['mpg'].values

In [ ]:
# Compute the observed sample mean
sample_mean = np.mean(mpg_data)

# Nonparametric bootstrap function to generate replicates of the mean
def bootstrap_sample(data, func, B=1000, seed=None):
    if seed is not None:
        np.random.seed(seed)  # Set the seed if provided
    n = len(data)
    bootstrap_stats = np.zeros(B)
    for i in range(B):
        resample = np.random.choice(data, size=n, replace=True)
        bootstrap_stats[i] = func(resample)
    return bootstrap_stats

# Generate bootstrap replicates for the mean with a fixed seed
bootstrap_means = bootstrap_sample(mpg_data, np.mean, B=1000, seed=123)

# Calculate the bootstrap standard error
se_theta_hat = np.std(bootstrap_means, ddof=1)  # Bootstrap standard error estimate

# Set the confidence level
alpha = 0.05
z_alpha = norm.ppf(1 - alpha / 2)

# Construct the standard normal bootstrap confidence interval using bootstrap SE

normal_ci = (sample_mean - z_alpha * se_theta_hat,
             sample_mean + z_alpha * se_theta_hat)

print("Standard Normal Bootstrap Confidence Interval for the Mean:", normal_ci)

Standard Normal Bootstrap Confidence Interval for the Mean: (np.float64(22.67858074963484), np.float64(24.213255985059035))


### Basic Bootstrap Confidence Interval

The **Basic Bootstrap Confidence Interval**, also known as the **Pivotal Confidence Interval**, adjusts the bootstrap distribution by centering it around the observed statistic. In this approach, we transform the distribution of the bootstrap replicates by subtracting the observed estimate, $\hat{\theta}$. The confidence interval is then constructed using quantiles of the shifted distribution, specifically $\hat{\theta}^* - \hat{\theta}$.

For a $100(1 - \alpha)\%$ confidence interval, the bounds are given by

\begin{equation*}
\left(2 \hat{\theta} - \hat{\theta}_{1 - \alpha / 2}^*, \quad 2 \hat{\theta} - \hat{\theta}_{\alpha / 2}^*\right),
\end{equation*}

where $\hat{\theta}_{\alpha}^*$ denotes the $\alpha$ quantile of the bootstrap replicates $\hat{\theta}^*$. This interval uses the idea that the difference $\hat{\theta} - \theta$ can be approximated by the quantiles of $\hat{\theta}^* - \hat{\theta}$, allowing us to estimate the distribution of the error around the observed statistic.


In [61]:
bootstrap_means = bootstrap_sample(mpg_data, np.mean, seed=123)

# Calculate quantiles for the basic bootstrap confidence interval
alpha = 0.05
lower_bound = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
upper_bound = np.percentile(bootstrap_means, 100 * (alpha / 2))

# Compute the basic bootstrap confidence interval

basic_ci = (2*sample_mean - lower_bound, 2*sample_mean - upper_bound)

print("Basic Bootstrap Confidence Interval for the Mean:", basic_ci)


Basic Bootstrap Confidence Interval for the Mean: (np.float64(22.693341836734692), np.float64(24.2110331632653))


### Percentile Bootstrap Confidence Interval

The **Percentile Bootstrap Confidence Interval** uses the empirical distribution of the bootstrap replicates as the reference distribution. By directly using the quantiles of this empirical distribution, we can estimate the sampling distribution of the statistic $\hat{\theta}$ without assuming normality. Efron and Tibshirani show that this interval has theoretical advantages over the standard normal interval and generally offers better coverage accuracy.

For a $100(1 - \alpha)\%$ confidence interval, we calculate the $\alpha / 2$ and $1 - \alpha / 2$ quantiles of the bootstrap replicates, denoted $\hat{\theta}_{\alpha / 2}^*$ and $\hat{\theta}_{1 - \alpha / 2}^*$, respectively. This interval thus captures the central range of the bootstrap estimates and provides a robust confidence interval for non-normally distributed statistics.



In [62]:
bootstrap_means = bootstrap_sample(mpg_data, np.mean, seed=123)

# Set the confidence level
alpha = 0.05

# Calculate the Percentile Bootstrap Confidence Interval

percentile_ci = (np.percentile(bootstrap_means, 100 * (alpha / 2)),
                 np.percentile(bootstrap_means, 100 * (1 - alpha / 2)))

print("95% Percentile Bootstrap Confidence Interval for the Mean:", percentile_ci)

95% Percentile Bootstrap Confidence Interval for the Mean: (np.float64(22.680803571428577), np.float64(24.198494897959183))
